## Clean Movie Reviews


**Movie Reviews/Genres/emotions dataset**

In [317]:
import sys
!{sys.executable} -m pip install beautifulsoup4

You should consider upgrading via the '/Users/alexandraflores/venv/.venv/bin/python -m pip install --upgrade pip' command.


In [318]:
pip install seaborn

You should consider upgrading via the '/Users/alexandraflores/venv/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [319]:
pip install pandas ipykernel

You should consider upgrading via the '/Users/alexandraflores/venv/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import pandas
import pandas as pd

# Additional libraries for text cleaning
import re
from bs4 import BeautifulSoup

# Read a CSV file
df = pd.read_csv("/Users/alexandraflores/Desktop/IDMb movie reviews:genres.csv", encoding="utf-8")

# View the first few records
df.head()

ModuleNotFoundError: No module named 'bs4'

In [321]:
# Inspect the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46173 entries, 0 to 46172
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   46173 non-null  int64 
 1   Ratings      46173 non-null  int64 
 2   Reviews      46173 non-null  object
 3   movie_name   46173 non-null  object
 4   Resenhas     46173 non-null  object
 5   genres       46173 non-null  object
 6   Description  46173 non-null  object
 7   emotion      46173 non-null  object
dtypes: int64(2), object(6)
memory usage: 2.8+ MB


** IMDb Movies Dataset**

In [322]:
# Read a CSV file
df_movie2 = pd.read_csv("/Users/alexandraflores/Desktop/imdb_movies.csv", encoding="utf-8")

# View the first few records
df_movie2.head()

,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,3/2/23,73,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/22,78,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,4/5/23,76,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,Mummies,1/5/23,70,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,Supercell,3/17/23,61,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US


In [ ]:
# View the first 10 records
df_movie2.head(10)

### COMBINE BOTH DATASETS

In [ ]:
df_combined = pd.merge(
    df,
    df_movie2,
    left_on="movie_name",
    right_on="names",
    how="inner"
)

In [ ]:
df_combined.head(10)

In [ ]:
# Keep only the columns we need
df_combined = df_combined[["Ratings", "Reviews", "movie_name", "genres","Description","emotion","overview","budget_x","revenue"]]


df_combined.head()

In [ ]:
# Check missing values
df_combined.isnull().sum()

In [ ]:
# Remove exact duplicate records
df = df.drop_duplicates()

df.shape

In [ ]:
# Make sure reviews, description, emotion stored as strings
df_combined["Reviews"] = df_combined["Reviews"].astype(str)


df_combined["Description"] = df_combined["Description"].astype(str)


df_combined["emotion"] = df_combined["emotion"].astype(str)

df_combined["movie_name"] = df_combined["movie_name"].astype(str)


df_combined["overview"] = df_combined["overview"].astype(str)




In [ ]:
# Remove HTML tags from text
def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

In [ ]:
# Remove URLs from text
def remove_urls(text):
    return re.sub(r"http\S+|www\S+|https\S+", "", text)

In [ ]:
# Replace multiple spaces, tabs, and line breaks with one space
def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# Apply basic cleaning steps
def basic_clean_text(text):
    #text = fix_mojibake(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = normalize_whitespace(text)
    text = text.lower()
    return text

In [ ]:
# Apply cleaning function
df_combined["clean_text"] = df_combined["Description"].apply(basic_clean_text)

df_combined[["Description", "clean_text"]].head()

In [ ]:
# Apply cleaning function
df_combined["clean_text2"] = df_combined["Reviews"].apply(basic_clean_text)

df_combined[["Reviews", "clean_text2"]].head()


In [ ]:
# Apply cleaning function
df_combined["clean_text3"] = df_combined["emotion"].apply(basic_clean_text)

df_combined[["emotion", "clean_text3"]].head()


In [ ]:
# Apply cleaning function
df_combined["clean_text4"] = df_combined["movie_name"].apply(basic_clean_text)

df_combined[["movie_name", "clean_text4"]].head()

In [ ]:
# Apply cleaning function
df_combined["clean_text5"] = df_combined["overview"].apply(basic_clean_text)

df_combined[["overview", "clean_text5"]].head()

In [ ]:
# Remove empty text after cleaning
df_combined = df_combined[df_combined["clean_text"].str.len() > 0]
df_combined = df_combined[df_combined["clean_text2"].str.len() > 0]
df_combined = df_combined[df_combined["clean_text3"].str.len() > 0]
df_combined = df_combined[df_combined["clean_text4"].str.len() > 0]
df_combined = df_combined[df_combined["clean_text5"].str.len() > 0]

In [ ]:
import sys
!{sys.executable} -m pip install langdetect

In [ ]:
#!pip install langdetect

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0

def is_english(text):
    """
    Return True if the text is detected as English.
    """
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

# test
print(is_english("Appartement cosy et idéalement situé"))  # should return False
print(is_english("Great place, highly recommend!"))  # should return True

# Keep only English reviews
# df = df[df["clean_text"].apply(is_english)]
# Processing 1.7 million reviews with the standard langdetect library will take approximately 3.5 to 6 hours on a single CPU. Because the library is single-threaded in Python, computing exactly 1.7 million rows of ~100 words typically averages around 10-12 milliseconds per sample.

In [ ]:
duplicates = df_combined.duplicated(
    subset=[
        "clean_text",
        "clean_text2",
        "clean_text3",
        "clean_text4",
        "clean_text5"
    ]
).sum()
print(f"Duplicate rows: {duplicates}")

In [ ]:
# Remove duplicate text after normalization
df_combined = df_combined.drop_duplicates(
    subset=[
        "clean_text",
        "clean_text2",
        "clean_text3",
        "clean_text4",
        "clean_text5"
    ]
)

df_combined.shape

duplicates = df_combined.duplicated(
    subset=[
        "clean_text",
        "clean_text2",
        "clean_text3",
        "clean_text4",
        "clean_text5"
    ]
).sum()


In [ ]:
# Add simple text length features
df_combined["char_count"] = df_combined["clean_text"].str.len()
df_combined["word_count"] = df_combined["clean_text"].str.split().str.len()

df_combined[["clean_text"]].head()

In [ ]:
# Add simple text length features
df_combined["char_count"] = df_combined["clean_text2"].str.len()
df_combined["word_count"] = df_combined["clean_text2"].str.split().str.len()

df_combined[["clean_text2", "char_count", "word_count"]].head()

In [ ]:
# Add simple text length features
df_combined["char_count"] = df_combined["clean_text3"].str.len()
df_combined["word_count"] = df_combined["clean_text3"].str.split().str.len()

df_combined[["clean_text3", "char_count", "word_count"]].head()

In [ ]:
# Add simple text length features
df_combined["char_count"] = df_combined["clean_text4"].str.len()
df_combined["word_count"] = df_combined["clean_text4"].str.split().str.len()

df_combined[["clean_text4", "char_count", "word_count"]].head()

In [ ]:
# Add simple text length features
df_combined["char_count"] = df_combined["clean_text5"].str.len()
df_combined["word_count"] = df_combined["clean_text5"].str.split().str.len()

df_combined[["clean_text5", "char_count", "word_count"]].head()

### CLEAN GENRES


In [ ]:
df_combined['genres'].value_counts()

In [ ]:
import ast
df_combined["genres"] = (
    df_combined["genres"]
    .apply(ast.literal_eval)
    .str[0]
)

In [ ]:
df_combined.head()

In [ ]:
review_counts = df_combined['movie_name'].value_counts()
review_counts.describe()
review_counts.head(10)

In [ ]:
# Save cleaned dataset
df_combined.to_csv("movie_reviews_clean.csv", index=False, encoding="utf-8-sig") 

In [ ]:
print(df_combined.columns)

In [ ]:
# Remove very short reviews
df_combined = df_combined[df_combined["word_count"] >= 5]

In [ ]:
df_combined.shape

In [ ]:
# Sample about 1,000 records for the course dataset
df_sample = df_combined.sample(n=1000, random_state=42)

df_sample.shape

In [ ]:
df_combined.head()

In [ ]:
df_sample.columns.tolist()

In [ ]:
# Select final columns
df_final = df_sample[
    [
        "Ratings",
        "Reviews",
        "movie_name",
        "genres",
        "Description",
        "emotion",
        "overview",
        "budget_x",
        "revenue",
        "clean_text",
        "clean_text2",
        "clean_text3",
        "clean_text4",
        "clean_text5",
        "char_count",
        "word_count"
    ]
]

df_final.head()

### EDA VISUALIZATION


In [ ]:
pip install nltk

In [ ]:
import ssl
import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")


In [ ]:
pip install spaCy

In [ ]:
# Install libraries not included in Colab by default
!pip install wordcloud -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk import FreqDist, pos_tag

# spaCy
import spacy
import subprocess, sys
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], capture_output=True)
nlp = spacy.load('en_core_web_sm')

# WordCloud
from wordcloud import WordCloud

STOP_WORDS = set(stopwords.words('english'))
print("All libraries loaded.")

custom_stopwords = {'movie', 'film','like','really','much','would','could','well','way','movies','films'}
STOP_WORDS = STOP_WORDS.union(custom_stopwords)



In [ ]:
import sys
print(sys.executable)

## LOAD DATA

In [ ]:
# Read a CSV file

df = pd.read_csv("movie_reviews_clean.csv", encoding="utf-8")
df_sample = df.sample(n=1000, random_state=42)
# Our main text column to list
texts = df_sample['clean_text2'].astype(str).tolist()

print(f"Rows loaded : {len(df_sample)}")
print(f"Columns     : {list(df_sample.columns)}")
df_sample.head(10)

## PRE COMPUTE TOKENS & SENTENCES

In [ ]:
all_words = []   # list of word lists, one per document
all_sents = []   # list of sentence lists, one per document

for text in texts:
    tokens = word_tokenize(text.lower())
    words  = [w for w in tokens if w.isalpha()]
    sents  = sent_tokenize(text)
    all_words.append(words)
    all_sents.append(sents)

print(f"Documents processed: {len(all_words)}")
print(f"Sample tokens (doc 0): {all_words[0][:10]}")

### Corpus Overview

In [ ]:
n_docs = len(texts)

# Flatten all words into one list
flat_words = []
for word_list in all_words:
    for word in word_list:
        flat_words.append(word)

total_words = len(flat_words)
total_chars = sum(len(text) for text in texts)

print(f"Number of documents : {n_docs:,}")
print(f"Total words         : {total_words:,}")
print(f"Total characters    : {total_chars:,}")

## Document length

In [ ]:
word_counts = []
for word_list in all_words:
    word_counts.append(len(word_list))

avg_length = sum(word_counts) / len(word_counts)

print(f"Average document length : {avg_length:.1f} words")
print(f"Shortest document       : {min(word_counts)} words")
print(f"Longest document        : {max(word_counts)} words")

### Vocab and Lexical Diversity

In [ ]:
vocab         = set(flat_words)
vocab_size    = len(vocab)
lexical_div   = vocab_size / total_words

print(f"Vocabulary size   : {vocab_size:,} unique words")
print(f"Lexical diversity : {lexical_div:.4f}")

### Text Structure Statistics

In [ ]:
char_counts = []
for text in texts:
    char_counts.append(len(text))

sent_counts = []
for sent_list in all_sents:
    sent_counts.append(len(sent_list))

print("Word counts  — mean:", round(np.mean(word_counts), 1),
      " median:", round(np.median(word_counts), 1))
print("Char counts  — mean:", round(np.mean(char_counts), 1),
      " median:", round(np.median(char_counts), 1))
print("Sent counts  — mean:", round(np.mean(sent_counts), 1),
      " median:", round(np.median(sent_counts), 1))

### Average Words Per Sentence

In [ ]:
avg_wps = []   # average words per sentence, one value per document
for i in range(len(texts)):
    if sent_counts[i] == 0:
        avg_wps.append(0)
    else:
        avg_wps.append(word_counts[i] / sent_counts[i])

print(f"Average words per sentence (corpus mean): {np.mean(avg_wps):.1f}")

## Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric'  : ['Word Count', 'Char Count', 'Sentence Count', 'Avg Words/Sent'],
    'Mean'    : [np.mean(word_counts), np.mean(char_counts),
                 np.mean(sent_counts), np.mean(avg_wps)],
    'Median'  : [np.median(word_counts), np.median(char_counts),
                 np.median(sent_counts), np.median(avg_wps)],
    'Std Dev' : [np.std(word_counts), np.std(char_counts),
                 np.std(sent_counts), np.std(avg_wps)],
    'Min'     : [min(word_counts), min(char_counts),
                 min(sent_counts), min(avg_wps)],
    'Max'     : [max(word_counts), max(char_counts),
                 max(sent_counts), max(avg_wps)],
})
summary = summary.round(2)
summary

### Length Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(word_counts, bins=40, color='steelblue', edgecolor='white')
plt.axvline(np.mean(word_counts), color='red', linestyle='--',
            label=f"Mean: {np.mean(word_counts):.0f}")
plt.title('Distribution of Word Counts per Document')
plt.xlabel('Word Count')
plt.ylabel('Number of Documents')
plt.legend()
plt.tight_layout()
plt.show()

###  Histogram of Character Counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(char_counts, bins=40, color='darkorange', edgecolor='white')
plt.axvline(np.mean(char_counts), color='red', linestyle='--',
            label=f"Mean: {np.mean(char_counts):.0f}")
plt.title('Distribution of Character Counts per Document')
plt.xlabel('Character Count')
plt.ylabel('Number of Documents')
plt.legend()
plt.tight_layout()
plt.show()

### Histogram of Sentence Counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(sent_counts, bins=30, color='mediumseagreen', edgecolor='white')
plt.axvline(np.mean(sent_counts), color='red', linestyle='--',
            label=f"Mean: {np.mean(sent_counts):.0f}")
plt.title('Distribution of Sentence Counts per Document')
plt.xlabel('Sentence Count')
plt.ylabel('Number of Documents')
plt.legend()
plt.tight_layout()
plt.show()

###  Identify Short and Long Listings

In [ ]:
p10 = np.percentile(word_counts, 10)
p90 = np.percentile(word_counts, 90)

short_listings = []
long_listings  = []

for i in range(len(texts)):
    if word_counts[i] <= p10:
        short_listings.append(i)
    elif word_counts[i] >= p90:
        long_listings.append(i)

print(f"10th percentile : {p10:.0f} words  ->  {len(short_listings)} short listings")
print(f"90th percentile : {p90:.0f} words  ->  {len(long_listings)} long listings")

print("\n--- Sample SHORT listing ---")
print(texts[short_listings[0]][:300])

print("\n--- Sample LONG listing ---")
print(texts[long_listings[0]][:300])

###  word count vs price

In [ ]:
import sys
!{sys.executable} -m pip install scipy

In [ ]:
pip install numpy

In [ ]:
import numpy as np
from scipy import stats

# Build the arrays
desc_word_counts = []
revenue         = []

for i in range(len(texts)):
    desc_word_counts.append(word_counts[i])
    revenue.append(df['revenue'].iloc[i])

# Calculate trendline (linear regression)
slope, intercept, r_value, p_value, std_err = stats.linregress(desc_word_counts, revenue)

trendline_x = list(range(min(desc_word_counts), max(desc_word_counts) + 1))
trendline_y = []
for x in trendline_x:
    trendline_y.append(slope * x + intercept)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(desc_word_counts, revenue, alpha=0.4, color='steelblue', edgecolors='none', s=30)
plt.plot(trendline_x, trendline_y, color='red', linewidth=2, label=f'Trendline (r = {r_value:.2f})')

plt.title('Description Word Count vs Revenue')
plt.xlabel('Word Count')
plt.ylabel('Revenue (USD)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Slope     : {slope:.4f}  (revenue change per extra word)")
print(f"R-squared : {r_value**2:.4f}  (how much word count explains revenue)")
print(f"P-value   : {p_value:.4f}  ({'significant' if p_value < 0.05 else 'not significant'})")

### Word Frequency Analysis

Top Words with Stopwords

In [ ]:
# Count every word in the corpus
freq_all = Counter(flat_words)

top_all = freq_all.most_common(20)
print("Top 20 words (including stopwords):")
for word, count in top_all:
    print(f"  {word:<20} {count}")

### Top Content Words (stopwords removed)

In [ ]:
flat_content = []
for word in flat_words:
    if word not in STOP_WORDS:
        flat_content.append(word)

freq_content = Counter(flat_content)

top_content = freq_content.most_common(20)
print("Top 20 content words (no stopwords):")
for word, count in top_content:
    print(f"  {word:<20} {count}")

### Bar Charts: Top Words vs Content Words

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all words
words_all, counts_all = zip(*top_all)
axes[0].barh(list(reversed(words_all)), list(reversed(counts_all)), color='steelblue')
axes[0].set_title('Top 20 Words (with stopwords)')
axes[0].set_xlabel('Frequency')

# Right: content words
words_c, counts_c = zip(*top_content)
axes[1].barh(list(reversed(words_c)), list(reversed(counts_c)), color='darkorange')
axes[1].set_title('Top 20 Content Words (no stopwords)')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Get the unique price categories to see what values exist
print(df_sample['revenue'].unique())
print(df_sample['revenue'].value_counts())

### Frequency Distribution 

In [ ]:
all_counts_sorted = sorted(freq_content.values(), reverse=True)
ranks = list(range(1, len(all_counts_sorted) + 1))

plt.figure(figsize=(8, 4))
plt.plot(ranks, all_counts_sorted, color='purple')
plt.xscale('log')
plt.yscale('log')
plt.title("Word Frequency Distribution (log-log) — Zipf's Law")
plt.xlabel('Rank')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### Word Cloud

In [ ]:
text_for_cloud = ' '.join(flat_content)

wc = WordCloud(width=900, height=400, background_color='white',
               max_words=150, colormap='tab10').generate(text_for_cloud)

plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Content Words')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 content words by price category

# Separate texts by price category
comedy = []
romance  = []

for i in range(len(texts)):
    category = str(df_sample['genres'].iloc[i]).lower()
    for word in all_words[i]:
        if word not in STOP_WORDS:
            if "comedy" in category:
                comedy.append(word)
            elif "romance" in category:
                romance.append(word)

from collections import Counter
import matplotlib.pyplot as plt

# Get top 20 words
top_comedy = Counter(comedy).most_common(20)
top_romance = Counter(romance).most_common(20)

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High-price listings
words_h, counts_h = zip(*top_comedy)

axes[0].barh(
    list(reversed(words_h)),
    list(reversed(counts_h)),
    color='firebrick'
)

axes[0].set_title('Comedy Reviews')
axes[0].set_xlabel('Frequency')

# Low-price listings
words_l, counts_l = zip(*top_romance)

axes[1].barh(
    list(reversed(words_l)),
    list(reversed(counts_l)),
    color='steelblue'
)

axes[1].set_title('Romance Reviews')
axes[1].set_xlabel('Frequency')

plt.suptitle(
    'Top 20 Content Words by Genre',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
E

In [ ]:
# Top 20 content words by genre, COMEDY and DRAMA

# Separate texts by genre
comedy = []
drama  = []

for i in range(len(texts)):
    category = str(df_sample['genres'].iloc[i]).lower()
    for word in all_words[i]:
        if word not in STOP_WORDS:
            if "comedy" in category:
                comedy.append(word)
            elif "drama" in category:
                drama.append(word)

from collections import Counter
import matplotlib.pyplot as plt

# Get top 20 words
top_comedy = Counter(comedy).most_common(20)
top_drama = Counter(drama).most_common(20)

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High-price listings
words_h, counts_h = zip(*top_comedy)

axes[0].barh(
    list(reversed(words_h)),
    list(reversed(counts_h)),
    color='firebrick'
)

axes[0].set_title('Comedy Reviews')
axes[0].set_xlabel('Frequency')

# Low-price listings
words_l, counts_l = zip(*top_drama)

axes[1].barh(
    list(reversed(words_l)),
    list(reversed(counts_l)),
    color='steelblue'
)

axes[1].set_title('Drama Reviews')
axes[1].set_xlabel('Frequency')

plt.suptitle(
    'Top 20 Content Words by Genre',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# Build word clouds
wc_comedy = WordCloud(width=800, height=350, background_color='white', max_words=150, colormap='Reds').generate(' '.join(comedy))

wc_drama = WordCloud(width=800, height=350, background_color='white',
                       max_words=150, colormap='Blues').generate(' '.join(drama))

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(wc_comedy, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Comedy Reviews', fontsize=13)

axes[1].imshow(wc_drama, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Drama Reviews', fontsize=13)

plt.suptitle('Word Clouds by Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 content words by GENRE comedy-action

# Separate texts by price category
comedy = []
action  = []

for i in range(len(texts)):
    category = str(df_sample['genres'].iloc[i]).lower()
    for word in all_words[i]:
        if word not in STOP_WORDS:
            if "comedy" in category:
                comedy.append(word)
            elif "action" in category:
                action.append(word)

from collections import Counter
import matplotlib.pyplot as plt

# Get top 20 words
top_comedy = Counter(comedy).most_common(20)
top_action = Counter(action).most_common(20)

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High-price listings
words_h, counts_h = zip(*top_comedy)

axes[0].barh(
    list(reversed(words_h)),
    list(reversed(counts_h)),
    color='firebrick'
)

axes[0].set_title('Comedy Reviews')
axes[0].set_xlabel('Frequency')

# Low-price listings
words_l, counts_l = zip(*top_action)

axes[1].barh(
    list(reversed(words_l)),
    list(reversed(counts_l)),
    color='steelblue'
)

axes[1].set_title('Action Reviews')
axes[1].set_xlabel('Frequency')

plt.suptitle(
    'Top 20 Content Words by Genre',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# Build word clouds - comedy - action
wc_comedy = WordCloud(width=800, height=350, background_color='white', max_words=150, colormap='Reds').generate(' '.join(comedy))

wc_action = WordCloud(width=800, height=350, background_color='white',
                       max_words=150, colormap='Oranges').generate(' '.join(action))

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(wc_comedy, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Comedy Reviews', fontsize=13)

axes[1].imshow(wc_action, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Action Reviews', fontsize=13)

plt.suptitle('Word Clouds by Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Phrase Analysis

### -Extract Biagrams

In [ ]:
bigram_counter = Counter()

for word_list in all_words:
    # Remove stopwords
    content_words = []
    for word in word_list:
        if word not in STOP_WORDS:
            content_words.append(word)
    # Build bigrams: each consecutive pair
    for i in range(len(content_words) - 1):
        bigram = (content_words[i], content_words[i+1])
        bigram_counter[bigram] += 1

top_bigrams = bigram_counter.most_common(20)
print("Top 20 bigrams:")
for bg, count in top_bigrams:
    print(f"  {' '.join(bg):<30} {count}")

In [323]:
trigram_counter = Counter()

for word_list in all_words:
    content_words = []
    for word in word_list:
        if word not in STOP_WORDS:
            content_words.append(word)
    # Build trigrams: each consecutive triple
    for i in range(len(content_words) - 2):
        trigram = (content_words[i], content_words[i+1], content_words[i+2])
        trigram_counter[trigram] += 1

top_trigrams = trigram_counter.most_common(20)
print("Top 20 trigrams:")
for tg, count in top_trigrams:
    print(f"  {' '.join(tg):<40} {count}")

Top 20 trigrams:
  west side story                          17
  hunchback notre dame                     13
  helena bonham carter                     12
  happened one night                       10
  worst ever made                          10
  version beauty beast                     7
  one best seen                            7
  robert de niro                           6
  opening half hour                        6
  victor hugo novel                        6
  never let go                             6
  two main characters                      6
  main hoon hero                           6
  hoon hero tera                           6
  sets special effects                     5
  based true story                         5
  see fairy tale                           5
  around world days                        5
  girl next door                           5
  sacha baron cohen                        5


### Bar Charts: Top Bigrams and Trigrams 

In [1]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bigrams
bg_labels = [' '.join(bg) for bg, _ in top_bigrams]
bg_counts  = [c for _, c in top_bigrams]
axes[0].barh(list(reversed(bg_labels)), list(reversed(bg_counts)), color='teal')
axes[0].set_title('Top 20 Bigrams')
axes[0].set_xlabel('Frequency')

# Trigrams
tg_labels = [' '.join(tg) for tg, _ in top_trigrams]
tg_counts  = [c for _, c in top_trigrams]
axes[1].barh(list(reversed(tg_labels)), list(reversed(tg_counts)), color='coral')
axes[1].set_title('Top 20 Trigrams')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### Linguistic Preview (spaCy)

### Run spaCy Pipeline


In [ ]:
SAMPLE_N     = 300
sample_texts = texts[:SAMPLE_N]

pos_counter      = Counter()
ner_counter      = Counter()
ner_type_counter = Counter()
ner_high         = Counter()   # named entities for high price listings
ner_low          = Counter()   # named entities for low price listings

for i, doc in enumerate(nlp.pipe(sample_texts, batch_size=50, disable=['parser'])):
    category = df['genres'].iloc[i]
    # POS tags
    for token in doc:
        if token.is_alpha:
            pos_counter[token.pos_] += 1
    # Named entities
    for ent in doc.ents:
        entity = ent.text.lower()
        ner_counter[entity] += 1
        ner_type_counter[ent.label_] += 1
        # Split by price category
        if entity.isalpha() or ' ' in entity:
            if "comedy" in category:
                ner_high[entity] += 1
            elif "drama" in category:
                ner_low[entity] += 1

print(f"Finished processing {SAMPLE_N} documents.")
print(f"POS types found    : {len(pos_counter)}")
print(f"Entity types found : {len(ner_type_counter)}")
print(f"Comedy entities: {len(ner_high)}")
print(f"Drama entities : {len(ner_low)}")

In [ ]:
pos_labels = {
    'NOUN': 'Noun', 'VERB': 'Verb', 'ADJ': 'Adjective', 'ADV': 'Adverb',
    'PROPN': 'Proper Noun', 'ADP': 'Preposition', 'DET': 'Determiner',
    'PRON': 'Pronoun', 'AUX': 'Auxiliary', 'CCONJ': 'Coord. Conj.',
    'NUM': 'Number', 'PUNCT': 'Punctuation',
}

pos_names  = []
pos_counts = []
for tag, count in pos_counter.most_common(10):
    label = pos_labels.get(tag, tag)
    pos_names.append(label)
    pos_counts.append(count)

plt.figure(figsize=(8, 5))
plt.barh(list(reversed(pos_names)), list(reversed(pos_counts)), color='slateblue')
plt.title('POS Tag Distribution (top 10)')
plt.xlabel('Token Count')
plt.tight_layout()
plt.show()

print("\nPOS counts:")
for name, count in zip(pos_names, pos_counts):
    print(f"  {name:<18} {count:,}")

In [ ]:
# Compare word frequencies between high-price and low-price listings

from collections import Counter
import pandas as pd

high_freq = Counter(comedy)
low_freq  = Counter(drama)

# Create combined vocabulary
all_terms = set(high_freq.keys()) | set(low_freq.keys())

comparison = []

for term in all_terms:

    high_count = high_freq.get(term, 0)
    low_count  = low_freq.get(term, 0)

    difference = high_count - low_count

    comparison.append([
        term,
        high_count,
        low_count,
        difference
    ])

comparison_df = pd.DataFrame(
    comparison,
    columns=[
        "word",
        "high_count",
        "low_count",
        "difference"
    ]
)

In [ ]:

top_high_diff = (
    comparison_df
    .sort_values("difference", ascending=False)
    .head(20)
)

top_high_diff

### NLTK & spaCY 

###  Stemmming with NLTK

In [ ]:
from nltk.stem.porter import PorterStemmer
from nltk.stem.snowball import SnowballStemmer

porter   = PorterStemmer()
snowball = SnowballStemmer("english")

# Use top 20 content words from corpus (flat_content built in Section 4b)
top_words = [word for word, count in freq_content.most_common(20)]

print(f"{'Original':<20} {'Porter':<20} {'Snowball':<20}")
print("-" * 60)
for word in top_words:
    print(f"{word:<20} {porter.stem(word):<20} {snowball.stem(word):<20}")

### Lemmatization with NLTK

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()

print(f"{'Original':<20} {'Lemma (noun)':<20} {'Lemma (verb)':<20}")
print("-" * 60)
for word in top_words:
    print(f"{word:<20} {lemmatizer.lemmatize(word):<20} {lemmatizer.lemmatize(word, pos='v'):<20}")

###  Noun Chunks

In [ ]:
from collections import Counter

chunk_counter = Counter()

for doc in nlp.pipe(texts, batch_size=50):
    for chunk in doc.noun_chunks:
        chunk_text = chunk.text.lower().strip()
        if len(chunk_text.split()) > 1:   # only multi-word chunks
            chunk_counter[chunk_text] += 1

top_chunks = chunk_counter.most_common(20)

print("Top 20 noun chunks across all listings:")
for chunk, count in top_chunks:
    print(f"  {chunk:<40} {count}")

Text Representatoin and Embeddings

In [ ]:
!pip install -q gensim transformers torch scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
#Install libraries (run once)
!pip install -q gensim transformers torch scikit-learn pandas numpy matplotlib seaborn

In [ ]:
pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 is fast and high quality — great for demos
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded. Embedding dimension: {sbert.get_sentence_embedding_dimension()}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded')

### tf-idf

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    min_df=1
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text2'].astype(str))
tfidf_vocab  = tfidf_vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

In [ ]:
df_sample.head()

In [ ]:

# Combine all text fields as one would in practice
df_sample['full_text'] = df_sample['genres'] + ' ' + df_sample['clean_text2'] + ' ' + df_sample['clean_text4']

print(f'Dataset: {len(df_sample)} reviews')
df_sample[['genres','clean_text2','clean_text4']].head()

In [ ]:
# ── Our running example listing ──────────────────────────────
EXAMPLE_IDX = 0   # Reviews
example = df_sample.loc[EXAMPLE_IDX]

print('=' * 65)
print('RUNNING EXAMPLE LISTING')
print('=' * 65)
print(f"Genre       : {example['genres']}")
print(f"Reviews: {example['clean_text2']}...")
print(f"Movie Name     : {example['clean_text4']}")
print('=' * 65)

In [ ]:
# ── Our running example listing ──────────────────────────────
EXAMPLE_IDX2 = 1  # Reviews
example2 = df.loc[EXAMPLE_IDX2]

print('=' * 65)
print('RUNNING EXAMPLE LISTING')
print('=' * 65)
print(f"Genre       : {example2['genres']}")
print(f"Reviews: {example2['clean_text2']}...")
print(f"Movie Name     : {example2['clean_text4']}")
print('=' * 65)

In [ ]:
# Encode all listing descriptions
bert_vecs = sbert.encode(df['clean_text2'].tolist(), show_progress_bar=True)
print(f'\nBERT listing matrix shape: {bert_vecs.shape}')
print(f'BERT vector for father of the bride 2 -comedy- (first 10 dims):')
print(np.round(bert_vecs[EXAMPLE_IDX, :10], 3))

In [ ]:
# TF-IDF scores for the running example
example_tfidf = tfidf_matrix[EXAMPLE_IDX].toarray().flatten()
tfidf_df = pd.DataFrame({'word': tfidf_vocab, 'tfidf': example_tfidf})
tfidf_df = tfidf_df[tfidf_df['tfidf'] > 0].sort_values('tfidf', ascending=False)

print('TF-IDF scores for father of the bride 2 -comedy- (top words):')
print(tfidf_df.head(12).to_string(index=False))